# EXP-016 원인 분석: 조건 목록 RRF를 빼면 multi-doc이 왜 떨어지는가

EXP-016은 dev-full 순위가 EXP-015와 같고(판정 보류, +0.43%p), multi-doc만 EXP-015 대비 떨어졌다(Recall@5 −0.76%p, law·questioner). 같은 검색기에서 `conditions.fuse`만 바꿔 A안(EXP-015)과 B안(EXP-016)을 다시 만들고, B안에서 상위 5개 밖으로 밀려난 정답 문서가 조건에 몇 필드 맞는 문서였는지 나눈다. 질의당 지연도 같은 자리에서 잰다(`docs/experiments/EXP-016.md` 분석 절).

In [7]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import open_dict

from rag.conditions import build_extractor
from rag.eval_multi.metrics import load_gold, ranked_docs, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever, condition_docs
from rag.search import query_embeddings, query_files, query_list, read_texts

In [8]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256'
TOP = 5

## dev-full 조건 질의 수

dev-full 순위가 EXP-015와 같았다. 조건 목록(최다 일치 필드 수 ≥ `min_fields`)이 생기는 질의가 몇 개인지 센다.

In [9]:
cfg = load_cfg(['+exp=exp016', 'index.scope=full'])
ret = Retriever(cfg)
ex = build_extractor(cfg)
min_fields = cfg.retriever.conditions.min_fields
qids_full, path_full = query_list(cfg)
texts_full = read_texts(path_full, qids_full)
n_cond = 0
for t in texts_full:
    counts = ret.cond_index.match_counts(ex.extract(t))
    n_cond += bool(counts) and max(counts.values()) >= min_fields
print('dev-full 조건 질의', n_cond, '/', len(texts_full))

dev-full 조건 질의 4 / 3016


## multi-doc: A안·B안 다시 만들기

`conditions.fuse`만 바꿔 질의마다 두 안의 상위 10개를 만들고, 순위 파일(EXP-015, EXP-016)과 같은지 확인한다. 같은 자리에서 `rank` 호출 시간을 질의마다 잰다(인덱스 로딩 제외).

In [10]:
mcfg = load_cfg(['+exp=exp016', 'index.scope=full', 'search.queries=multidoc'])
gold = load_gold(ROOT / mcfg.paths.multidoc_frozen)
qids, text_path = query_list(mcfg)
texts = read_texts(text_path, qids)
emb = query_embeddings(mcfg, qids, text_path, query_files(mcfg)['emb'])
tokens = ret.query_tokens(texts)
conds = [ex.extract(t) for t in texts]
top = {True: {}, False: {}}
secs = {True: [], False: []}
for fuse in (True, False):
    with open_dict(cfg):
        cfg.retriever.conditions.fuse = fuse
    for j, q in enumerate(qids):
        t0 = time.perf_counter()
        docs = ret.rank(ret.chunk_emb @ emb[j], tokens[j], conds[j])
        secs[fuse].append(time.perf_counter() - t0)
        top[fuse][q] = docs
files = {True: f'{PREFIX}-cc-multidoc.csv', False: f'{PREFIX}-cc-nocl-multidoc.csv'}
for fuse, f in files.items():
    run = ranked_docs(read_run(RUNS / f), gold, 10)
    same = sum(run[q][:10] == top[fuse][q] for q in qids)
    print('fuse', fuse, '순위 파일과 같은 질의', same, '/', len(qids))
ms = pd.DataFrame({('A' if k else 'B'): pd.Series(v) * 1000 for k, v in secs.items()})
print(ms.describe().loc[['mean', '50%', 'max']].round(1), '(ms)')

fuse True 순위 파일과 같은 질의 600 / 600
fuse False 순위 파일과 같은 질의 600 / 600
          A      B
mean  129.6  126.6
50%   118.1  114.0
max   355.1  365.9 (ms)


## 밀려난 정답 문서의 조건 일치

정답 문서마다 A안·B안 상위 5개 안 여부를 보고, 맞은 필드 수로 나눈다. best는 최다 일치(필터가 앞으로 올림), list는 조건 목록에 들지만 최다 일치가 아닌 문서(A안에서만 RRF로 순위를 얻음), none은 조건 목록 밖이다.

In [11]:
rows = []
for j, q in enumerate(qids):
    counts = ret.cond_index.match_counts(conds[j])
    if counts and max(counts.values()) >= min_fields:
        row = ret.chunk_emb @ emb[j]
        d = np.full(len(ret.doc_names), -np.inf, dtype=row.dtype)
        np.maximum.at(d, ret.chunk_doc, row)
        cond = set(condition_docs(counts, d, ret.doc_names, ret.depth, min_fields))
        best = max(counts.values())
    else:
        cond, best = set(), None
    by_name = {ret.doc_names[i]: c for i, c in counts.items()}
    for g in gold.loc[q, 'gold']:
        kind = ('best' if by_name.get(g) == best else 'list') if g in cond else 'none'
        rows.append({'qid': q, 'type': gold.loc[q, 'type'], 'doc': g, 'kind': kind,
                     'in_a': g in top[True][q][:TOP], 'in_b': g in top[False][q][:TOP]})
gd = pd.DataFrame(rows)
gd['move'] = np.select([gd.in_a & ~gd.in_b, ~gd.in_a & gd.in_b], ['lost', 'gained'], 'same')
print(pd.crosstab([gd['type'], gd['kind']], gd['move']))

move             lost  same
type       kind            
conf       best     0   540
           list     1     7
law        best     0   340
           list     4    40
questioner best     1   543
           list     5    10
